In [5]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, RepeatVector, TimeDistributed
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Step 1: Load the dataset from CSV
# Replace 'freelancer_profiles.csv' with the path to your actual CSV file
df = pd.read_csv('modified_freelancer.csv')

# Verify the dataset
print("Dataset Info:")
print(df.info())
print("\nFirst few rows:")
print(df.head())

# Step 2: Preprocess the Data Separately for Bio and Skills
# Bio Preprocessing
max_vocab_size_bio = 10000
max_len_bio = int(np.percentile([len(str(x).split()) for x in df['bio']], 95))  # Dynamic max length for bio
tokenizer_bio = Tokenizer(num_words=max_vocab_size_bio, lower=True, oov_token='<OOV>')
tokenizer_bio.fit_on_texts(df['bio'])
sequences_bio = tokenizer_bio.texts_to_sequences(df['bio'])
padded_sequences_bio = pad_sequences(sequences_bio, maxlen=max_len_bio, padding='post', truncating='post')
X_bio = np.array(padded_sequences_bio)

# Skills Preprocessing
max_vocab_size_skills = 5000
max_len_skills = int(np.percentile([len(str(x).split(',')) for x in df['skills']], 95))  # Dynamic max length for skills
tokenizer_skills = Tokenizer(num_words=max_vocab_size_skills, lower=True, oov_token='<OOV>', split=',')
tokenizer_skills.fit_on_texts(df['skills'])
sequences_skills = tokenizer_skills.texts_to_sequences(df['skills'])
padded_sequences_skills = pad_sequences(sequences_skills, maxlen=max_len_skills, padding='post', truncating='post')
X_skills = np.array(padded_sequences_skills)

# Step 3: Split into Training and Validation Sets
X_train_bio, X_val_bio = train_test_split(X_bio, test_size=0.2, random_state=42)
X_train_skills, X_val_skills = train_test_split(X_skills, test_size=0.2, random_state=42)

# Step 4: Define the Autoencoder Model (with Explicit Input Shape)
def build_autoencoder(input_dim, max_len, embedding_dim=64, latent_dim=32):
    model = Sequential([
        # Encoder
        Embedding(input_dim=input_dim, output_dim=embedding_dim, input_length=max_len),
        LSTM(128, activation='relu', return_sequences=False),
        Dense(latent_dim, activation='relu'),  # Latent space bottleneck
        
        # Decoder
        RepeatVector(max_len),
        LSTM(128, activation='relu', return_sequences=True),
        TimeDistributed(Dense(input_dim, activation='softmax'))
    ])
    
    # Explicitly build the model to ensure parameters are calculated
    model.build(input_shape=(None, max_len))
    
    model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy')
    return model

# Build the autoencoders
bio_autoencoder = build_autoencoder(input_dim=max_vocab_size_bio, max_len=max_len_bio)
skills_autoencoder = build_autoencoder(input_dim=max_vocab_size_skills, max_len=max_len_skills)

# Print model summaries (should now show parameter counts)
print("Bio Autoencoder Summary:")
bio_autoencoder.summary()
print("\nSkills Autoencoder Summary:")
skills_autoencoder.summary()

# Step 5: Create tf.data.Dataset for Efficient Batching
batch_size = 64
train_dataset_bio = tf.data.Dataset.from_tensor_slices((X_train_bio, X_train_bio)).batch(batch_size)
val_dataset_bio = tf.data.Dataset.from_tensor_slices((X_val_bio, X_val_bio)).batch(batch_size)

train_dataset_skills = tf.data.Dataset.from_tensor_slices((X_train_skills, X_train_skills)).batch(batch_size)
val_dataset_skills = tf.data.Dataset.from_tensor_slices((X_val_skills, X_val_skills)).batch(batch_size)

# Step 6: Train the Models
epochs = 50

# Train Bio Autoencoder
print("Training Bio Autoencoder...")
bio_autoencoder.fit(
    train_dataset_bio,
    validation_data=val_dataset_bio,
    epochs=epochs,
    verbose=1
)

# Train Skills Autoencoder
print("Training Skills Autoencoder...")
skills_autoencoder.fit(
    train_dataset_skills,
    validation_data=val_dataset_skills,
    epochs=epochs,
    verbose=1
)

# Step 7: Evaluate Reconstruction Error (MSE)
# Bio
reconstructed_val_bio = bio_autoencoder.predict(X_val_bio)
mse_bio = tf.keras.losses.mean_squared_error(X_val_bio, reconstructed_val_bio.argmax(axis=-1)).numpy().mean()
print(f"Bio Reconstruction MSE: {mse_bio:.4f}")

# Skills
reconstructed_val_skills = skills_autoencoder.predict(X_val_skills)
mse_skills = tf.keras.losses.mean_squared_error(X_val_skills, reconstructed_val_skills.argmax(axis=-1)).numpy().mean()
print(f"Skills Reconstruction MSE: {mse_skills:.4f}")

# Step 8: Post-Processing (Convert Sequences Back to Text)
def sequences_to_text(sequences, tokenizer):
    inverse_word_index = {v: k for k, v in tokenizer.word_index.items()}
    texts = []
    for seq in sequences:
        text = ' '.join([inverse_word_index.get(word, '<OOV>') for word in seq if word != 0])
        texts.append(text)
    return texts

# Example prediction for Bio
sample_input_bio = X_val_bio[:1]
enhanced_output_bio = bio_autoencoder.predict(sample_input_bio)
enhanced_seq_bio = enhanced_output_bio.argmax(axis=-1)
original_text_bio = sequences_to_text(sample_input_bio, tokenizer_bio)
enhanced_text_bio = sequences_to_text(enhanced_seq_bio, tokenizer_bio)

# Example prediction for Skills
sample_input_skills = X_val_skills[:1]
enhanced_output_skills = skills_autoencoder.predict(sample_input_skills)
enhanced_seq_skills = enhanced_output_skills.argmax(axis=-1)
original_text_skills = sequences_to_text(sample_input_skills, tokenizer_skills)
enhanced_text_skills = sequences_to_text(enhanced_seq_skills, tokenizer_skills)

print(f"Original Bio: {original_text_bio[0]}")
print(f"Enhanced Bio: {enhanced_text_bio[0]}")
print(f"Original Skills: {original_text_skills[0]}")
print(f"Enhanced Skills: {enhanced_text_skills[0]}")



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   skills  10000 non-null  object
 1   bio     10000 non-null  object
dtypes: object(2)
memory usage: 156.4+ KB
None

First few rows:
                                         skills  \
0  color palette, adobe lightroom, print design   
1                                sql, analytics   
2      data mining, statistics, data processing   
3   sql programming, database architecture, sql   
4      responsive design, user interface design   

                                                 bio  
0  Senior Graphic & Visual Design professional wi...  
1  Mid-level Data Science professional with 4 yea...  
2  Entry-level Data Science professional with 2 y...  
3  Senior Database Management professional with 1...  
4  Senior UI/UX Design professional with 10 years...  


C:\Users\Siddhesh Patil\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Bio Autoencoder Summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ (None, 38, 64)              │         640,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 128)                 │          98,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 32)                  │           4,128 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ repeat_vector (RepeatVector)         │ (None, 38, 32)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 38, 128)             │          82,432 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ time_distributed (TimeDistributed)   │ (None, 38, 10000)           │       1,290,000 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

...

 Total params: 1,150,376 (4.39 MB)

 Trainable params: 1,150,376 (4.39 MB)

 Non-trainable params: 0 (0.00 B)

Training Bio Autoencoder...
Epoch 1/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 81s 515ms/step - loss: 7.5012 - val_loss: 4.1391
Epoch 2/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 65s 522ms/step - loss: 3.6243 - val_loss: 2.5260
Epoch 3/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 62s 499ms/step - loss: 2.5058 - val_loss: 2.8688
Epoch 4/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 62s 499ms/step - loss: 2.1937 - val_loss: 1.9302
Epoch 5/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 63s 506ms/step - loss: 2.1619 - val_loss: 1.7274
Epoch 6/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 62s 497ms/step - loss: 1.9640 - val_loss: 1.9732
Epoch 7/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 62s 496ms/step - loss: 1.8941 - val_loss: 1.7361
Epoch 8/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 62s 492ms/step - loss: 1.7208 - val_loss: 1.6702
Epoch 9/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 63s 504ms/step - loss: 1.6595 - val_loss: 1.5637
Epoch 10/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 62s 492ms/step - loss: 1.6006 - val_loss: 1.5492
Epoch 11/50
125/125 ━━━━━━━━━━━━━━━━━━━━ 62s 497ms/step - loss: 1.7411 - val_loss

AttributeError: module 'keras._tf_keras.keras.losses' has no attribute 'mean_squared_error'

In [16]:
# Step 7: Evaluate Reconstruction Error (MSE)
# Bio
reconstructed_val_bio = bio_autoencoder.predict(X_val_bio)
mse_fn = tf.keras.losses.MeanSquaredError()
mse_bio = mse_fn(X_val_bio, reconstructed_val_bio.argmax(axis=-1)).numpy()
print(f"Bio Reconstruction MSE: {mse_bio:.4f}")

# Skills
reconstructed_val_skills = skills_autoencoder.predict(X_val_skills)
mse_skills = mse_fn(X_val_skills, reconstructed_val_skills.argmax(axis=-1)).numpy()
print(f"Skills Reconstruction MSE: {mse_skills:.4f}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 67ms/step
Bio Reconstruction MSE: 4574.7368
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step
Skills Reconstruction MSE: 8407.1602


In [10]:
pip install mean_squared_error

Note: you may need to restart the kernel to use updated packages.


In [14]:
# Step 8: Post-Processing (Convert Sequences Back to Text)
def sequences_to_text(sequences, tokenizer):
    inverse_word_index = {v: k for k, v in tokenizer.word_index.items()}
    texts = []
    for seq in sequences:
        text = ' '.join([inverse_word_index.get(word, '<OOV>') for word in seq if word != 0])
        texts.append(text)
    return texts



In [15]:
# Example prediction for Bio
sample_input_bio = X_val_bio[:1]
enhanced_output_bio = bio_autoencoder.predict(sample_input_bio)
enhanced_seq_bio = enhanced_output_bio.argmax(axis=-1)
original_text_bio = sequences_to_text(sample_input_bio, tokenizer_bio)
enhanced_text_bio = sequences_to_text(enhanced_seq_bio, tokenizer_bio)

# Example prediction for Skills
sample_input_skills = X_val_skills[:1]
enhanced_output_skills = skills_autoencoder.predict(sample_input_skills)
enhanced_seq_skills = enhanced_output_skills.argmax(axis=-1)
original_text_skills = sequences_to_text(sample_input_skills, tokenizer_skills)
enhanced_text_skills = sequences_to_text(enhanced_seq_skills, tokenizer_skills)

print(f"Original Bio: {original_text_bio[0]}")
print(f"Enhanced Bio: {enhanced_text_bio[0]}")
print(f"Original Skills: {original_text_skills[0]}")
print(f"Enhanced Skills: {enhanced_text_skills[0]}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 835ms/step
Original Bio: mid level business consulting professional with 4 years of experience specializing in business management business valuation relationship management brings deep expertise in business intelligence software proficient in developing scalable solutions and optimizing workflows
Enhanced Bio: mid level business consulting professional with 5 years of experience specializing in business management business business business brings brings deep expertise in in business plan proficient in developing scalable solutions and optimizing workflows
Original Skills: business management  business valuation  relationship management  business intelligence software
Enhanced Skills: business management  corporate social responsibility  business services  strategy


In [17]:
# Step 7: Evaluate Reconstruction Error (MSE)
# Bio
reconstructed_val_bio = bio_autoencoder.predict(X_val_bio)
mse_bio = np.mean((X_val_bio - reconstructed_val_bio.argmax(axis=-1)) ** 2)
print(f"Bio Reconstruction MSE: {mse_bio:.4f}")

# Skills
reconstructed_val_skills = skills_autoencoder.predict(X_val_skills)
mse_skills = np.mean((X_val_skills - reconstructed_val_skills.argmax(axis=-1)) ** 2)
print(f"Skills Reconstruction MSE: {mse_skills:.4f}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 64ms/step
Bio Reconstruction MSE: 4574.7366
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
Skills Reconstruction MSE: 8407.1604


In [19]:
plt.figure(figsize=(10, 5))
plt.plot(bio.history['loss'], label='Training Loss (Bio)')
plt.plot(bio.history['val_loss'], label='Validation Loss (Bio)')
plt.title('Bio Autoencoder: Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss (Sparse Categorical Crossentropy)')
plt.legend()
plt.grid()
plt.show()

NameError: name 'bio' is not defined

<Figure size 1000x500 with 0 Axes>